# Mount And Start Using mdfs

This notebook shows the optional OS mount path using the `mdfs-mount` binary.

The mount binary is behind the Rust `fuser` feature. On macOS you need macFUSE and `pkg-config` available so `fuser` can link against the real mount library. If your machine is not configured for FUSE, use `01_getting_started_http.ipynb` first; the HTTP path is the recommended agent integration path today.

## 1. Build The Mount Binary

On macOS, install prerequisites with something like `brew install macfuse pkg-config` if this build cannot find FUSE.

In [ ]:
from pathlib import Path
import atexit
import os
import platform
import shutil
import subprocess
import time


def find_repo_root(start=None):
    """Find the mdfs repo even when Jupyter starts in examples/notebooks."""
    env_root = os.environ.get("MDFS_REPO_ROOT") or os.environ.get("MARKDOWNFS_REPO_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())

    start_path = Path(start or Path.cwd()).resolve()
    candidates.extend([start_path, *start_path.parents])

    for candidate in candidates:
        if (candidate / "Cargo.toml").exists() and (candidate / "src").is_dir():
            return candidate

    raise RuntimeError(
        "Could not find the mdfs repository root. "
        "Run the notebook from inside the repo or set MDFS_REPO_ROOT."
    )


def find_cargo():
    """Find Cargo even when the Jupyter kernel has a minimal PATH."""
    candidates = []
    env_cargo = os.environ.get("CARGO")
    if env_cargo:
        candidates.append(Path(env_cargo).expanduser())

    path_cargo = shutil.which("cargo")
    if path_cargo:
        candidates.append(Path(path_cargo))

    candidates.extend([
        Path.home() / ".cargo" / "bin" / "cargo",
        Path("/opt/homebrew/bin/cargo"),
        Path("/usr/local/bin/cargo"),
    ])

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return str(candidate)

    raise RuntimeError(
        "Could not find Cargo. Install Rust from https://rustup.rs, "
        "or set CARGO=/absolute/path/to/cargo before running this notebook."
    )


def find_pkg_config():
    """Find pkg-config even when the Jupyter kernel has a minimal PATH."""
    candidates = []
    env_pkg_config = os.environ.get("PKG_CONFIG")
    if env_pkg_config:
        candidates.append(Path(env_pkg_config).expanduser())

    path_pkg_config = shutil.which("pkg-config")
    if path_pkg_config:
        candidates.append(Path(path_pkg_config))

    candidates.extend([
        Path("/opt/homebrew/bin/pkg-config"),
        Path("/usr/local/bin/pkg-config"),
        Path("/opt/local/bin/pkg-config"),
    ])

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return str(candidate)

    raise RuntimeError(
        "Could not find pkg-config. On macOS with Homebrew, run `brew install pkg-config`, "
        "or set PKG_CONFIG=/absolute/path/to/pkg-config before running this notebook."
    )


def add_common_fuse_pkg_config_paths(env):
    """Help pkg-config find macFUSE when Jupyter has a minimal environment."""
    candidates = [
        Path("/opt/homebrew/lib/pkgconfig"),
        Path("/usr/local/lib/pkgconfig"),
        Path("/usr/local/share/pkgconfig"),
        Path("/opt/local/lib/pkgconfig"),
        Path("/Library/Filesystems/macfuse.fs/Contents/Resources/lib/pkgconfig"),
    ]
    existing = [str(path) for path in candidates if path.exists()]
    current = env.get("PKG_CONFIG_PATH", "")
    if current:
        existing.append(current)
    if existing:
        env["PKG_CONFIG_PATH"] = os.pathsep.join(existing)


ROOT = find_repo_root()
CARGO = find_cargo()
PKG_CONFIG = find_pkg_config()
DEMO_ROOT = ROOT / ".demo" / "notebook-mount"
DATA_DIR = DEMO_ROOT / "data"
MOUNTPOINT = DEMO_ROOT / "mnt"
print(f"Using mdfs repo at {ROOT}")
print(f"Using cargo at {CARGO}")
print(f"Using pkg-config at {PKG_CONFIG}")

DEMO_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MOUNTPOINT.mkdir(parents=True, exist_ok=True)

cargo_env = os.environ.copy()
cargo_env["PKG_CONFIG"] = PKG_CONFIG
cargo_env["PATH"] = os.pathsep.join([
    str(Path(CARGO).parent),
    str(Path(PKG_CONFIG).parent),
    cargo_env.get("PATH", ""),
])
add_common_fuse_pkg_config_paths(cargo_env)

build = subprocess.run(
    [CARGO, "build", "--release", "--features", "fuser", "--bin", "mdfs-mount"],
    cwd=ROOT,
    env=cargo_env,
    text=True,
    capture_output=True,
)
print(build.stdout)
print(build.stderr)
MOUNT_BUILD_OK = build.returncode == 0
MOUNT_BINARY = ROOT / "target" / "release" / "mdfs-mount"

if not MOUNT_BUILD_OK:
    print(
        "Mount build skipped: mdfs-mount could not be built on this machine.\n"
        "On macOS, install macFUSE and pkg-config so fuser can find fuse.pc.\n"
        "For Homebrew, try: brew install pkg-config && brew install --cask macfuse.\n"
        "If macFUSE is installed elsewhere, set PKG_CONFIG_PATH to the directory containing fuse.pc.\n"
        "You can still use 01_getting_started_http.ipynb without FUSE."
    )
else:
    print(f"Built mdfs-mount at {MOUNT_BINARY}")

## 2. Start The Mount

This starts `target/release/mdfs-mount` in the background. Keep the process alive while you use the mounted directory.

In [ ]:
mount_proc = None
MOUNT_ACTIVE = False

def unmount():
    if mount_proc is None:
        return
    if platform.system() == "Darwin":
        subprocess.run(["diskutil", "unmount", str(MOUNTPOINT)], capture_output=True, text=True)
    else:
        subprocess.run(["fusermount", "-u", str(MOUNTPOINT)], capture_output=True, text=True)
    if mount_proc.poll() is None:
        mount_proc.terminate()
        try:
            mount_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            mount_proc.kill()

if not MOUNT_BUILD_OK:
    print("Skipping mount start because mdfs-mount was not built. See the previous cell for prerequisites.")
else:
    env = os.environ.copy()
    env.update({
        "MARKDOWNFS_DATA_DIR": str(DATA_DIR),
        "MARKDOWNFS_MOUNTPOINT": str(MOUNTPOINT),
    })
    env["PATH"] = os.pathsep.join([
        str(Path(CARGO).parent),
        str(Path(PKG_CONFIG).parent),
        env.get("PATH", ""),
    ])

    mount_proc = subprocess.Popen(
        [str(MOUNT_BINARY), "--mountpoint", str(MOUNTPOINT)],
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    atexit.register(unmount)

    time.sleep(2)
    if mount_proc.poll() is not None:
        output = mount_proc.stdout.read() if mount_proc.stdout else ""
        print(f"Mount process exited early. FUSE may not be installed or permitted.\n{output}")
    else:
        MOUNT_ACTIVE = True
        print(f"Mounted markdownfs at {MOUNTPOINT}")

## 3. Use The Mounted Directory

`mdfs` is markdown-only by design, so create `.md` files.

In [ ]:
if not MOUNT_ACTIVE:
    print("Skipping mounted-directory example because no FUSE mount is active.")
else:
    note = MOUNTPOINT / "hello.md"
    note.write_text("# Hello From A Mount\n\nThis markdown file was created through the OS mount.\n", encoding="utf-8")

    print(note.read_text(encoding="utf-8"))
    print(sorted(path.name for path in MOUNTPOINT.iterdir()))

## 4. Unmount

The mount binary uses an explicit unmount flow, so run this cell when you are done.

In [ ]:
if MOUNT_ACTIVE:
    unmount()
    MOUNT_ACTIVE = False
    print("unmounted")
else:
    print("No active mount to unmount.")